# Test Digit DataLoader

This notebook checks that `DigitDatasetModule` can: 
- find your dataset
- build train/val/test splits
- return valid batches
- visualize sample images

In [3]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt

In [4]:
# Ensure src is importable from this notebook location
notebook_dir = Path.cwd().resolve()
project_src = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir

if str(project_src) not in sys.path:
    sys.path.insert(0, str(project_src))

from data.datamodule import DigitDatasetModule

print('Using src path:', project_src)

Using src path: C:\Users\lovro\Desktop\hackatoni\transformer_architecture\src


In [5]:
# Change this if needed, or set DATA_ROOT_DIR in your environment
default_root = project_src / 'data' / 'digits'
data_root_dir = Path(os.getenv('DATA_ROOT_DIR', str(default_root))).resolve()

print('Data root:', data_root_dir)
print('Exists:', data_root_dir.exists())

Data root: C:\Users\lovro\Desktop\hackatoni\transformer_architecture\src\data\digits
Exists: False


In [6]:
if not data_root_dir.exists():
    raise FileNotFoundError(f'Dataset path does not exist: {data_root_dir}')

split_dirs = ['train', 'val', 'test']
present_splits = [s for s in split_dirs if (data_root_dir / s).exists()]

print('Detected split dirs:', present_splits if present_splits else 'none')
if present_splits and len(present_splits) != 3:
    print('Warning: partial split dirs detected. Provide all train/val/test or none.')

FileNotFoundError: Dataset path does not exist: C:\Users\lovro\Desktop\hackatoni\transformer_architecture\src\data\digits

In [ ]:
dm = DigitDatasetModule(
    data_root_dir=str(data_root_dir),
    batch_size=16,
    num_workers=0,
    image_size=28,
    val_split=0.1,
    test_split=0.1,
    seed=42,
    pin_memory=False,
    augment_train=True,
    image_mode='L',
)

dm.setup()
print('Train size:', len(dm.train_dataset))
print('Val size:', len(dm.valid_dataset))
print('Test size:', len(dm.test_dataset))
print('class_to_idx:', dm.train_dataset.class_to_idx)

In [ ]:
train_loader = dm.train_dataloader()
images, labels = next(iter(train_loader))

print('images shape:', tuple(images.shape))
print('labels shape:', tuple(labels.shape))
print('images dtype:', images.dtype)
print('labels dtype:', labels.dtype)

assert images.ndim == 4, 'Expected images shape [B, C, H, W]'
assert labels.ndim == 1, 'Expected labels shape [B]'

In [ ]:
n_show = min(8, images.shape[0])
fig, axes = plt.subplots(1, n_show, figsize=(2 * n_show, 2))
if n_show == 1:
    axes = [axes]

for i in range(n_show):
    img = images[i].detach().cpu()
    if img.shape[0] == 1:
        axes[i].imshow(img.squeeze(0), cmap='gray')
    else:
        axes[i].imshow(img.permute(1, 2, 0))
    axes[i].set_title(f'label={int(labels[i])}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()